# Queued Inference Demo (E2E vs Composed BDU-GRU)

This notebook runs both queue-based inference pipelines on one dataset selected by index:

- E2E BDU-GRU risk model
- Composed BDU-GRU risk model

Then it plots real risk vs predicted risk for both models over X frames.

In [11]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

from MIREIA.config import Config
from MIREIA.data_collection.dataset_utils import load_jsonl_records, resolve_image_path
from MIREIA.perception import (
    DepthAnythingV2Estimator,
    FeatureIntegrator,
    QueuedComposedBDUGRURiskInference,
    QueuedE2ERiskInference,
    create_environment_classifier_predictor,
    load_road_segmentation_model,
)

In [12]:
# --- Experiment config ---
source_jsonl_name = "dataset.jsonl"
dataset_index = 74          # Select dataset/scenario by index
start_frame = 0            # Starting frame index inside selected dataset
num_frames_to_plot = 3500    # X frames

# --- Model selection for inference + plot ---
model_selection = "both"   # "e2e", "composed", or "both"
selected_models = model_selection.strip().lower()
if selected_models not in {"e2e", "composed", "both"}:
    raise ValueError("model_selection must be one of: 'e2e', 'composed', 'both'")
run_e2e = selected_models in {"e2e", "both"}
run_composed = selected_models in {"composed", "both"}

# --- Discover datasets/scenarios by index ---
scenarios_root = Path(Config.PATH_TO_SCENARIOS)
scenario_dirs = [
    p for p in sorted(scenarios_root.iterdir())
    if p.is_dir() and p.name not in {"videos", "__pycache__"} and (p / source_jsonl_name).is_file()
]

if not scenario_dirs:
    raise RuntimeError(f"No scenarios with {source_jsonl_name} found under {scenarios_root}")

print("Available datasets (select by index):")
for i, scenario_dir in enumerate(scenario_dirs):
    print(f"[{i:02d}] {scenario_dir.name}")

if dataset_index < 0 or dataset_index >= len(scenario_dirs):
    raise ValueError(f"dataset_index={dataset_index} out of range [0, {len(scenario_dirs) - 1}]")

selected_scenario_dir = scenario_dirs[dataset_index]
records = load_jsonl_records(str(selected_scenario_dir / source_jsonl_name))
if not records:
    raise RuntimeError(f"No records found in {selected_scenario_dir / source_jsonl_name}")

end_frame = min(len(records), start_frame + int(num_frames_to_plot))
selected_records = records[start_frame:end_frame]
if not selected_records:
    raise RuntimeError("Selected frame range is empty.")

frame_paths = []
frame_ids = []
real_risks = []
for rec in selected_records:
    rel_image = str(rec.get("rgb_image_path", "")).strip()
    if not rel_image:
        continue

    full_path = resolve_image_path(str(selected_scenario_dir), rel_image, normalize_paths=True)
    if not os.path.isfile(full_path):
        continue

    frame_paths.append(full_path)
    frame_ids.append(int(rec.get("frame_id", len(frame_ids))))
    real_risks.append(float(rec.get("ground_truth_risk", 0.0)))

if not frame_paths:
    raise RuntimeError("No valid image paths found in selected frame range.")

print("\nSelected dataset:", selected_scenario_dir.name)
print("Usable frames:", len(frame_paths))
print("Model selection:", selected_models)

Available datasets (select by index):
[00] 01A_ClearNoon_Town01_HighVol
[01] 01B_ClearNoon_Town01_LowVol
[02] 01C_ClearNoon_Town02_HighVol
[03] 01D_ClearNoon_Town02_LowVol
[04] 02A_CloudyNoon_Town03_HighVol
[05] 02B_CloudyNoon_Town03_LowVol
[06] 02C_CloudyNoon_Town03_HighVol
[07] 02D_CloudyNoon_Town04_LowVol
[08] 03A_WetNoon_Town04_HighVol
[09] 03B_WetNoon_Town04_LowVol
[10] 03C_WetNoon_Town03_HighVol
[11] 03D_WetNoon_Town04_LowVol
[12] 04A_WetCloudyNoon_Town01_HighVol
[13] 04B_WetCloudyNoon_Town01_LowVol
[14] 04C_WetCloudyNoon_Town02_HighVol
[15] 04D_WetCloudyNoon_Town02_LowVol
[16] 05A_SoftRainNoon_Town03_HighVol
[17] 05B_SoftRainNoon_Town03_LowVol
[18] 05C_SoftRainNoon_Town01_HighVol
[19] 05D_SoftRainNoon_Town02_LowVol
[20] 06A_MidRainyNoon_Town04_HighVol
[21] 06B_MidRainyNoon_Town04_LowVol
[22] 06C_MidRainyNoon_Town03_HighVol
[23] 06D_MidRainyNoon_Town04_LowVol
[24] 07A_HardRainNoon_Town01_HighVol
[25] 07B_HardRainNoon_Town01_LowVol
[26] 07C_HardRainNoon_Town02_HighVol
[27] 07D_Har

In [13]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATHS = {
    "yolo": Path(Config.PATH_TO_MODELS) / "yolo11s.pt",
    "depth": Path(Config.PATH_TO_MODELS) / "depth_anything_v2_vits.pth",
    "climate": Path(Config.PATH_TO_MODELS) / "environment_multitask_checkpoint.pt",
    "road_seg": Path(Config.PATH_TO_MODELS) / "road_segmentation_multitask_checkpoint.pt",
}

# E2E checkpoint paths — add or remove entries to compare multiple runs
e2e_checkpoint_paths = {
    "e2e_latest": Path(Config.PATH_TO_MODELS) / "e2e_risk_checkpoint.pt",
    "e2e_1epoch": Path(Config.PATH_TO_MODELS) / "e2e_risk_checkpoint_1full_epoch.pt",
}

"""composed_checkpoint_paths = {
    f"bdu_search_{i:02d}": Path(Config.PATH_TO_MODELS) / f"bdu_gru_search_{i:02d}.pt"
    for i in range(1, 9)
}"""

composed_checkpoint_paths = { "bdu_gru_search_02": Path(Config.PATH_TO_MODELS) / "bdu_gru_search_02.pt" }

required = {}
if run_e2e:
    for name, path in e2e_checkpoint_paths.items():
        required[f"e2e_{name}"] = path
if run_composed:
    required.update({
        "yolo": MODEL_PATHS["yolo"],
        "depth": MODEL_PATHS["depth"],
    })
    for name, path in composed_checkpoint_paths.items():
        required[f"{name}_checkpoint"] = path

missing_required = [f"{name}: {path}" for name, path in required.items() if not path.is_file()]
if missing_required:
    raise FileNotFoundError("Missing required model file(s):\n" + "\n".join(missing_required))

e2e_predictors = {}
if run_e2e:
    for name, ckpt_path in e2e_checkpoint_paths.items():
        e2e_predictors[name] = QueuedE2ERiskInference.from_checkpoint(
            checkpoint_path=str(ckpt_path),
            device=device_name,
        )
# Alias for backward compat with the Town05 overlay cell
e2e_predictor = next(iter(e2e_predictors.values())) if e2e_predictors else None

composed_predictors = {}
if run_composed:
    yolo_model = YOLO(str(MODEL_PATHS["yolo"]))
    depth_estimator = DepthAnythingV2Estimator(
        checkpoint_path=MODEL_PATHS["depth"],
        encoder="vits",
        device=device_name,
    )

    environment_predictor = None
    if MODEL_PATHS["climate"].is_file():
        environment_predictor = create_environment_classifier_predictor(
            checkpoint_path=str(MODEL_PATHS["climate"]),
            device=device_name,
        )

    road_segmentation = None
    if MODEL_PATHS["road_seg"].is_file():
        road_segmentation = load_road_segmentation_model(
            checkpoint_path=str(MODEL_PATHS["road_seg"]),
            device=device_name,
        )

    for name, checkpoint_path in composed_checkpoint_paths.items():
        composed_predictors[name] = QueuedComposedBDUGRURiskInference.from_checkpoint(
            checkpoint_path=str(checkpoint_path),
            feature_integrator=FeatureIntegrator(),
            yolo_model=yolo_model,
            depth_estimator=depth_estimator,
            environment_predictor=environment_predictor,
            road_segmentation=road_segmentation,
            device=device_name,
        )

print("Loaded checkpoints:")
if run_e2e:
    for name, path in e2e_checkpoint_paths.items():
        print(f" - {name}: {path}")
if run_composed:
    for name, path in composed_checkpoint_paths.items():
        print(f" - {name}: {path}")


Loaded checkpoints:
 - e2e_latest: t:\TFG\MIREIA\models\e2e_risk_checkpoint.pt
 - e2e_1epoch: t:\TFG\MIREIA\models\e2e_risk_checkpoint_1full_epoch.pt
 - bdu_gru_search_02: t:\TFG\MIREIA\models\bdu_gru_search_02.pt


In [14]:
from time import perf_counter

used_frame_ids = [int(fid) for fid in frame_ids]
real_vals = [float(r) for r in real_risks]
total_frames = len(frame_paths)


def _run_model_inference(model_name, predictor):
    predictor.reset_queue()

    preds = []
    t_start = perf_counter()

    for done, (frame_path, frame_id) in enumerate(zip(frame_paths, used_frame_ids), start=1):
        out = predictor.add_image_path(frame_path)
        preds.append(float(out.latest_risk) if out.ready and out.latest_risk is not None else np.nan)

        if done % 10 == 0 or done == total_frames:
            elapsed_s = perf_counter() - t_start
            fps = done / max(elapsed_s, 1e-9)
            eta_s = (total_frames - done) / max(fps, 1e-9)
            print(
                f"[{model_name}] [{done}/{total_frames}] frame_id={frame_id} | "
                f"ready={out.ready} | fps={fps:.1f} eta={eta_s:.1f}s"
            )

    elapsed_total = perf_counter() - t_start
    fps_total = total_frames / max(elapsed_total, 1e-9)
    print(f"[{model_name}] finished in {elapsed_total:.1f}s ({fps_total:.1f} FPS).")
    return preds, elapsed_total, fps_total


e2e_preds_by_model = {}
e2e_elapsed_by_model = {}
e2e_fps_by_model = {}

composed_preds_by_model = {}
composed_elapsed_by_model = {}
composed_fps_by_model = {}

if run_e2e:
    print("Running E2E inference...")
    for ckpt_name, predictor in e2e_predictors.items():
        preds, elapsed_s, fps = _run_model_inference(f"E2E:{ckpt_name}", predictor)
        e2e_preds_by_model[ckpt_name] = preds
        e2e_elapsed_by_model[ckpt_name] = elapsed_s
        e2e_fps_by_model[ckpt_name] = fps
else:
    print("Skipping E2E inference (model_selection).")

if run_composed:
    print("\nRunning Composed BDU-GRU inference for all search checkpoints...")
    for ckpt_name, predictor in composed_predictors.items():
        preds, elapsed_s, fps = _run_model_inference(f"Composed:{ckpt_name}", predictor)
        composed_preds_by_model[ckpt_name] = preds
        composed_elapsed_by_model[ckpt_name] = elapsed_s
        composed_fps_by_model[ckpt_name] = fps
else:
    print("Skipping Composed inference (model_selection).")

# Backward-compat aliases used by the Town05 overlay cell
_e2e_default_key = next(iter(e2e_preds_by_model)) if e2e_preds_by_model else None
e2e_preds = e2e_preds_by_model[_e2e_default_key] if _e2e_default_key else [np.nan] * total_frames
e2e_elapsed_s = e2e_elapsed_by_model[_e2e_default_key] if _e2e_default_key else float("nan")
e2e_fps = e2e_fps_by_model[_e2e_default_key] if _e2e_default_key else float("nan")

default_composed_key = "bdu_search_01"
if run_composed and composed_preds_by_model:
    if default_composed_key not in composed_preds_by_model:
        default_composed_key = next(iter(composed_preds_by_model.keys()))
    composed_preds = composed_preds_by_model[default_composed_key]
    composed_elapsed_s = composed_elapsed_by_model[default_composed_key]
    composed_fps = composed_fps_by_model[default_composed_key]
else:
    composed_preds = [np.nan] * total_frames
    composed_elapsed_s = float("nan")
    composed_fps = float("nan")

print("\nFPS comparison:")
if run_e2e and e2e_fps_by_model:
    for ckpt_name, fps in e2e_fps_by_model.items():
        print(f" - E2E {ckpt_name}: {fps:.1f} FPS ({e2e_elapsed_by_model[ckpt_name]:.1f}s)")
if run_composed and composed_fps_by_model:
    for ckpt_name, fps in composed_fps_by_model.items():
        print(f" - Composed {ckpt_name}: {fps:.1f} FPS ({composed_elapsed_by_model[ckpt_name]:.1f}s)")
    avg_composed_fps = float(np.mean(list(composed_fps_by_model.values())))
    print(f" - Composed average: {avg_composed_fps:.1f} FPS")


Running E2E inference...
[E2E:e2e_latest] [10/3500] frame_id=9 | ready=False | fps=72.9 eta=47.9s
[E2E:e2e_latest] [20/3500] frame_id=19 | ready=True | fps=68.7 eta=50.7s
[E2E:e2e_latest] [30/3500] frame_id=29 | ready=True | fps=64.9 eta=53.5s
[E2E:e2e_latest] [40/3500] frame_id=39 | ready=True | fps=62.2 eta=55.7s
[E2E:e2e_latest] [50/3500] frame_id=49 | ready=True | fps=61.6 eta=56.0s
[E2E:e2e_latest] [60/3500] frame_id=59 | ready=True | fps=61.2 eta=56.2s
[E2E:e2e_latest] [70/3500] frame_id=69 | ready=True | fps=60.4 eta=56.8s
[E2E:e2e_latest] [80/3500] frame_id=79 | ready=True | fps=59.7 eta=57.3s
[E2E:e2e_latest] [90/3500] frame_id=89 | ready=True | fps=58.4 eta=58.4s
[E2E:e2e_latest] [100/3500] frame_id=99 | ready=True | fps=57.8 eta=58.8s
[E2E:e2e_latest] [110/3500] frame_id=109 | ready=True | fps=57.3 eta=59.1s
[E2E:e2e_latest] [120/3500] frame_id=119 | ready=True | fps=57.3 eta=59.0s
[E2E:e2e_latest] [130/3500] frame_id=129 | ready=True | fps=57.5 eta=58.6s
[E2E:e2e_latest] [1

In [ ]:
import matplotlib.patheffects as pe

def _masked_mse(pred: np.ndarray, target: np.ndarray) -> float:
    mask = np.isfinite(pred) & np.isfinite(target)
    if not np.any(mask):
        return float("nan")
    diff = pred[mask] - target[mask]
    return float(np.mean(diff * diff))

x = np.asarray(used_frame_ids, dtype=np.int64)
real_arr = np.asarray(real_vals, dtype=np.float32)

e2e_arr_by_model = (
    {name: np.asarray(preds, dtype=np.float32) for name, preds in e2e_preds_by_model.items()}
    if run_e2e
    else {}
)
composed_arr_by_model = (
    {name: np.asarray(preds, dtype=np.float32) for name, preds in composed_preds_by_model.items()}
    if run_composed
    else {}
)

e2e_mse_by_model = {
    name: _masked_mse(arr, real_arr)
    for name, arr in e2e_arr_by_model.items()
}
composed_mse_by_model = {
    name: _masked_mse(pred_arr, real_arr)
    for name, pred_arr in composed_arr_by_model.items()
}

# Plot mode options: "interactive_selector", "highlight_best", "show_all"
plot_mode = "interactive_selector"
default_checkpoint = "all"  # "all" or one of bdu_search_01..bdu_search_08

if run_composed and composed_mse_by_model:
    best_name = min(composed_mse_by_model, key=composed_mse_by_model.get)
else:
    best_name = None

_e2e_color_list = ["tab:blue", "tab:cyan", "tab:purple", "tab:olive"]

def _draw_plot(selected_checkpoint: str = "all") -> None:
    plt.figure(figsize=(14, 6))
    plt.plot(x, real_arr, label="Real Risk", linewidth=2.8, color="black")

    if run_e2e and e2e_arr_by_model:
        for idx, (name, arr) in enumerate(e2e_arr_by_model.items()):
            color = _e2e_color_list[idx % len(_e2e_color_list)]
            line, = plt.plot(
                x,
                arr,
                label=f"E2E {name}",
                linewidth=2.2,
                color=color,
            )
            line.set_path_effects([pe.Stroke(linewidth=4.0, foreground="white"), pe.Normal()])

    if run_composed and composed_arr_by_model:
        cmap = plt.cm.get_cmap("tab10")
        for idx, (name, pred_arr) in enumerate(composed_arr_by_model.items()):
            show_line = selected_checkpoint == "all" or selected_checkpoint == name
            if not show_line:
                continue

            is_emphasized = False
            if selected_checkpoint != "all":
                is_emphasized = name == selected_checkpoint
            elif plot_mode == "highlight_best" and best_name is not None:
                is_emphasized = name == best_name

            line_width = 2.8 if is_emphasized else 1.4
            line_alpha = 1.0 if is_emphasized else 0.7
            line_zorder = 5 if is_emphasized else 2

            line, = plt.plot(
                x,
                pred_arr,
                label=f"Composed {name}",
                linewidth=line_width,
                alpha=line_alpha,
                color=cmap(idx % 10),
                zorder=line_zorder,
            )

            if is_emphasized:
                line.set_path_effects([pe.Stroke(linewidth=line_width + 2.0, foreground="black"), pe.Normal()])

    if len(x) >= Config.INFERENCE_SEQUENCE_LENGTH:
        plt.axvline(
            x=x[Config.INFERENCE_SEQUENCE_LENGTH - 1],
            color="gray",
            linestyle="--",
            alpha=0.5,
            label="Queue Warmup Complete",
        )

    plt.xlabel("Frame ID")
    plt.ylabel("Risk")

    title_metrics = []
    for name, mse in e2e_mse_by_model.items():
        title_metrics.append(f"E2E {name} MSE={mse:.6f}")
    if run_composed and composed_mse_by_model and best_name is not None:
        title_metrics.append(f"Best {best_name} MSE={composed_mse_by_model[best_name]:.6f}")

    title = f"Risk Comparison | Dataset={selected_scenario_dir.name} | Frames={len(x)}"
    if selected_checkpoint != "all":
        title += f"\nSelected checkpoint: {selected_checkpoint}"
    if title_metrics:
        title += "\n" + " | ".join(title_metrics)
    plt.title(title)

    plt.legend(loc="upper right", ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

if run_composed and composed_arr_by_model and plot_mode == "interactive_selector":
    try:
        import ipywidgets as widgets
        from IPython.display import display

        checkpoint_options = ["all"] + sorted(composed_arr_by_model.keys())
        if default_checkpoint not in checkpoint_options:
            default_checkpoint = "all"

        selector = widgets.Dropdown(
            options=checkpoint_options,
            value=default_checkpoint,
            description="Checkpoint:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="320px"),
        )

        output_widget = widgets.interactive_output(
            _draw_plot,
            {"selected_checkpoint": selector},
        )

        display(widgets.VBox([selector, output_widget]))
    except Exception as exc:
        print(f"Interactive selector unavailable ({exc}). Falling back to static plot.")
        _draw_plot(default_checkpoint)
else:
    _draw_plot(default_checkpoint)

if run_e2e and e2e_mse_by_model:
    print("E2E MSE by checkpoint:")
    for name, mse in e2e_mse_by_model.items():
        print(f" - {name}: {mse:.6f}")
if run_composed and composed_mse_by_model:
    print("Composed MSE by checkpoint:")
    for name, mse in composed_mse_by_model.items():
        print(f" - {name}: {mse:.6f}")


E2E MSE by checkpoint:
 - e2e_latest: 5.559169
 - e2e_1epoch: 5.566879
Composed MSE by checkpoint:
 - bdu_gru_search_02: 4.926775


## Town05 Overlay (Sets 17 to 21)
Compare two Town05 scenarios from sets 17 to 21 only, with predicted risk overlaid on top of real risk (Town10 excluded).

In [18]:
import re
import matplotlib.patheffects as pe

# --- Town10 overlay config ---
overlay_max_frames = min(500, int(num_frames_to_plot))

# E2E checkpoints to include (keys from e2e_predictors)
overlay_e2e_keys = list(e2e_predictors.keys())
# Composed checkpoints to include (keys from composed_predictors / composed_checkpoint_paths)
overlay_composed_keys = ["bdu_gru_search_02"]

# ---

_slot_re = re.compile(r"^(?P<set>\d{2})(?P<letter>[A-D])_")

def _set_number(name: str) -> int | None:
    m = _slot_re.match(name)
    return int(m.group("set")) if m else None

def _is_town10(name: str) -> bool:
    return "town10" in name.lower()

def _load_scenario_series(scenario_dir: Path, max_frames: int):
    records = load_jsonl_records(str(scenario_dir / source_jsonl_name))
    paths, real = [], []
    for rec in records:
        rel = str(rec.get("rgb_image_path", "")).strip()
        if not rel:
            continue
        full = resolve_image_path(str(scenario_dir), rel, normalize_paths=True)
        if not os.path.isfile(full):
            continue
        paths.append(full)
        real.append(float(rec.get("ground_truth_risk", 0.0)))
        if len(paths) >= max_frames:
            break
    if not paths:
        raise RuntimeError(f"No valid image paths found for {scenario_dir.name}")
    return paths, np.asarray(real, dtype=np.float32)

def _predict_series(predictor, frame_paths: list, tag: str) -> np.ndarray:
    predictor.reset_queue()
    preds = []
    total = len(frame_paths)
    for idx, fp in enumerate(frame_paths, start=1):
        out = predictor.add_image_path(fp)
        preds.append(float(out.latest_risk) if out.ready and out.latest_risk is not None else np.nan)
        if idx % 100 == 0 or idx == total:
            print(f"[{tag}] {idx}/{total} frames")
    return np.asarray(preds, dtype=np.float32)

def _mse(pred: np.ndarray, target: np.ndarray) -> float:
    mask = np.isfinite(pred) & np.isfinite(target)
    if not np.any(mask):
        return float("nan")
    err = pred[mask] - target[mask]
    return float(np.mean(err * err))

# Load any composed checkpoints that weren't loaded in cell 3 (e.g. model_selection="e2e")
_missing_composed = [key for key in overlay_composed_keys if key not in composed_predictors]
if _missing_composed:
    print("Loading missing composed predictors on demand...")
    try:
        _yolo = yolo_model
    except NameError:
        _yolo = YOLO(str(Path(Config.PATH_TO_MODELS) / "yolo11s.pt"))
    try:
        _depth = depth_estimator
    except NameError:
        _depth = DepthAnythingV2Estimator(
            checkpoint_path=Path(Config.PATH_TO_MODELS) / "depth_anything_v2_vits.pth",
            encoder="vits",
            device=device_name,
        )
    try:
        _env = environment_predictor
    except NameError:
        _env = None
    try:
        _road = road_segmentation
    except NameError:
        _road = None
    for key in _missing_composed:
        if key not in composed_checkpoint_paths:
            print(f"Warning: '{key}' not in composed_checkpoint_paths, skipping.")
            continue
        ckpt_path = composed_checkpoint_paths[key]
        if not ckpt_path.is_file():
            print(f"Warning: checkpoint not found: {ckpt_path}, skipping.")
            continue
        composed_predictors[key] = QueuedComposedBDUGRURiskInference.from_checkpoint(
            checkpoint_path=str(ckpt_path),
            feature_integrator=FeatureIntegrator(),
            yolo_model=_yolo,
            depth_estimator=_depth,
            environment_predictor=_env,
            road_segmentation=_road,
            device=device_name,
        )
        print(f"  Loaded: {key} ({ckpt_path.name})")

# Build overlay predictor dict
overlay_predictors = {}
for key in overlay_e2e_keys:
    if key in e2e_predictors:
        overlay_predictors[f"e2e:{key}"] = e2e_predictors[key]
    else:
        print(f"Warning: e2e key '{key}' not found, skipping.")
for key in overlay_composed_keys:
    if key in composed_predictors:
        overlay_predictors[f"composed:{key}"] = composed_predictors[key]
    else:
        print(f"Warning: composed key '{key}' not found, skipping.")

if not overlay_predictors:
    raise RuntimeError("No overlay predictors available. Check overlay_e2e_keys and overlay_composed_keys.")

# Find Town10 scenarios
town10_candidates = sorted(
    [p for p in scenario_dirs if _is_town10(p.name)],
    key=lambda p: (_set_number(p.name) or 0, p.name),
)
if len(town10_candidates) < 2:
    raise RuntimeError(
        f"Need at least 2 Town10 scenarios. Found: {[p.name for p in town10_candidates]}"
    )

scenario_a = town10_candidates[0]
scenario_b = town10_candidates[-1]
if scenario_b == scenario_a:
    scenario_b = town10_candidates[1]

print(f"Scenario A (Town10): {scenario_a.name}")
print(f"Scenario B (Town10): {scenario_b.name}")

a_paths, a_real = _load_scenario_series(scenario_a, overlay_max_frames)
b_paths, b_real = _load_scenario_series(scenario_b, overlay_max_frames)

# Run inference for every overlay model on both scenarios
a_preds_by_model = {}
b_preds_by_model = {}
for model_name, predictor in overlay_predictors.items():
    print(f"\n--- {model_name} ---")
    a_preds_by_model[model_name] = _predict_series(predictor, a_paths, f"A:{model_name}")
    b_preds_by_model[model_name] = _predict_series(predictor, b_paths, f"B:{model_name}")

a_mse_by_model = {name: _mse(arr, a_real) for name, arr in a_preds_by_model.items()}
b_mse_by_model = {name: _mse(arr, b_real) for name, arr in b_preds_by_model.items()}

# Color assignment: E2E → blues, composed → oranges
_e2e_colors = ["tab:blue", "tab:cyan", "tab:purple", "tab:olive"]
_comp_colors = ["tab:orange", "tab:red", "tab:pink", "tab:brown"]
_overlay_colors = {}
_ei, _ci = 0, 0
for name in overlay_predictors:
    if name.startswith("e2e:"):
        _overlay_colors[name] = _e2e_colors[_ei % len(_e2e_colors)]
        _ei += 1
    else:
        _overlay_colors[name] = _comp_colors[_ci % len(_comp_colors)]
        _ci += 1

def _draw_overlay(selected_model: str = "all") -> None:
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, real, preds_by_model, mse_by_model, scenario_dir in [
        (ax_a, a_real, a_preds_by_model, a_mse_by_model, scenario_a),
        (ax_b, b_real, b_preds_by_model, b_mse_by_model, scenario_b),
    ]:
        x = np.arange(len(real), dtype=np.int64)
        ax.plot(x, real, label="Real Risk", linewidth=2.2, color="black")

        for name, arr in preds_by_model.items():
            if selected_model != "all" and selected_model != name:
                continue
            is_emphasized = selected_model == name
            color = _overlay_colors.get(name, "gray")
            lw = 2.4 if is_emphasized else 1.4
            alpha = 1.0 if is_emphasized else 0.65
            zorder = 5 if is_emphasized else 2
            mse_val = mse_by_model.get(name, float("nan"))
            line, = ax.plot(
                x, arr,
                label=f"{name} (MSE={mse_val:.5f})",
                linewidth=lw, color=color, alpha=alpha, zorder=zorder,
            )
            if is_emphasized:
                line.set_path_effects([pe.Stroke(linewidth=lw + 2.0, foreground="black"), pe.Normal()])

        ax.set_xlabel("Frame Index")
        ax.set_ylabel("Risk")
        ax.set_title(scenario_dir.name)
        ax.legend(fontsize=7, loc="upper right")

    suffix = f" — {selected_model}" if selected_model != "all" else ""
    plt.suptitle(f"Town10 Risk Overlay{suffix}", fontsize=12)
    plt.tight_layout()
    plt.show()

try:
    import ipywidgets as widgets
    from IPython.display import display

    model_options = ["all"] + list(overlay_predictors.keys())
    selector = widgets.Dropdown(
        options=model_options,
        value="all",
        description="Model:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="360px"),
    )
    output_widget = widgets.interactive_output(_draw_overlay, {"selected_model": selector})
    display(widgets.VBox([selector, output_widget]))
except Exception as exc:
    print(f"Interactive selector unavailable ({exc}). Falling back to static plot.")
    _draw_overlay("all")

print("\nMSE summary:")
for name in overlay_predictors:
    print(f"  {name}: A={a_mse_by_model[name]:.6f}  B={b_mse_by_model[name]:.6f}")


Scenario A (Town10): 17C_WetNoon_Town10HD_HighVol
Scenario B (Town10): 21D_HardRainNoon_Town10HD_LowVol_NoFog_Night

--- e2e:e2e_latest ---
[A:e2e:e2e_latest] 100/500 frames
[A:e2e:e2e_latest] 200/500 frames
[A:e2e:e2e_latest] 300/500 frames
[A:e2e:e2e_latest] 400/500 frames
[A:e2e:e2e_latest] 500/500 frames
[B:e2e:e2e_latest] 100/500 frames
[B:e2e:e2e_latest] 200/500 frames
[B:e2e:e2e_latest] 300/500 frames
[B:e2e:e2e_latest] 400/500 frames
[B:e2e:e2e_latest] 500/500 frames

--- e2e:e2e_1epoch ---
[A:e2e:e2e_1epoch] 100/500 frames
[A:e2e:e2e_1epoch] 200/500 frames
[A:e2e:e2e_1epoch] 300/500 frames
[A:e2e:e2e_1epoch] 400/500 frames
[A:e2e:e2e_1epoch] 500/500 frames
[B:e2e:e2e_1epoch] 100/500 frames
[B:e2e:e2e_1epoch] 200/500 frames
[B:e2e:e2e_1epoch] 300/500 frames
[B:e2e:e2e_1epoch] 400/500 frames
[B:e2e:e2e_1epoch] 500/500 frames

--- composed:bdu_gru_search_02 ---
[A:composed:bdu_gru_search_02] 100/500 frames
[A:composed:bdu_gru_search_02] 200/500 frames
[A:composed:bdu_gru_search_0


MSE summary:
  e2e:e2e_latest: A=7.113127  B=146.652039
  e2e:e2e_1epoch: A=5.988370  B=65.532013
  composed:bdu_gru_search_02: A=0.492825  B=21.253407
